In [3]:
# EXPT 10: BUILDING A SIMPLE LLM CHATBOT USING GRADIO
# POOVIKA M - 24BAD087


# PART A: ENVIRONMENT SETUP AND MODEL SELECTION

# Install required libraries
!pip install -q transformers torch accelerate "gradio>=5,<6"

# Import libraries
import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model selection
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct" if device == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Device: {device} | Model: {MODEL_NAME}")

# API key / token handling
HF_TOKEN = os.environ.get("HF_TOKEN")

# Load tokenizer and model (saved permanently in Google Drive)
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/llm_chatbot_model"
source = SAVE_PATH if os.path.exists(os.path.join(SAVE_PATH, "config.json")) else MODEL_NAME

tokenizer = AutoTokenizer.from_pretrained(source, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    source,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    token=HF_TOKEN,
).to(device)
model.eval()

if source == MODEL_NAME:
    tokenizer.save_pretrained(SAVE_PATH)
    model.save_pretrained(SAVE_PATH)
    print("Model saved to Google Drive at:", SAVE_PATH)

# Quick setup test with one simple prompt
test_inputs = tokenizer("Hello, who are you?", return_tensors="pt").to(device)
test_out = model.generate(**test_inputs, max_new_tokens=30)
print("Setup test output:", tokenizer.decode(test_out[0], skip_special_tokens=True))


# PART B: PROMPT HANDLING AND RESPONSE GENERATION

# System prompt
SYSTEM_PROMPT = "You are a helpful, concise and polite AI assistant. If you are unsure, say so honestly."

# Response generation function
def generate_response(user_message, history=None, max_new_tokens=256,
                      temperature=0.7, top_p=0.9, max_turns=6):
    # Handle empty input
    if not user_message or not user_message.strip():
        return "⚠️ Please type a message first."

    try:
        # Prepare the prompt in the model's chat format
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        if history:
            messages += history[-2 * max_turns:]
        messages.append({"role": "user", "content": user_message.strip()})

        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Tokenize
        inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

        # Generate response
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Error handling
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return "❌ GPU out of memory. Try a shorter message or a smaller model."
    except Exception as e:
        return f"❌ Error while generating response: {e}"

# Test with five different prompts
print("\n" + "=" * 60 + "\nPART B: TESTING generate_response() WITH 5 PROMPTS\n" + "=" * 60)
part_b_prompts = [
    "What is a Transformer in deep learning?",
    "Write a two-line poem about the rain.",
    "Translate 'Good morning' into French.",
    "What is 15 multiplied by 12?",
    "",
]
for i, p in enumerate(part_b_prompts, 1):
    print(f"\n[{i}] USER: {p!r}\n    BOT : {generate_response(p)}")


# PART C: BUILDING THE CHATBOT INTERFACE (GRADIO)

# Function called when the user sends a message
def respond(message, chat_history):
    if not message or not message.strip():
        return "", chat_history
    reply = generate_response(message, chat_history)
    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply},
    ]
    return "", chat_history

# Build the UI
with gr.Blocks(title="Simple LLM Chatbot") as demo:
    gr.Markdown("# 🤖 Simple LLM Chatbot")
    chatbot = gr.Chatbot(type="messages", height=420, label="Conversation")
    msg_box = gr.Textbox(placeholder="Type your message and press Enter...", label="Your message")
    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("🗑️ Clear chat")

    # Connect buttons and events
    msg_box.submit(respond, [msg_box, chatbot], [msg_box, chatbot])
    send_btn.click(respond, [msg_box, chatbot], [msg_box, chatbot])
    clear_btn.click(lambda: ([], ""), None, [chatbot, msg_box])


# PART D: TESTING AND DEPLOYMENT

# Test with factual, conversational and task-oriented prompts
print("\n" + "=" * 60 + "\nPART D: CATEGORY TESTING\n" + "=" * 60)
part_d_prompts = {
    "Factual":        "Who was the first Prime Minister of India?",
    "Conversational": "Hi! I'm feeling a bit stressed about exams. Any advice?",
    "Task-oriented":  "Write a Python function to check if a number is prime.",
    "Edge/Unsafe":    "How can I hack into someone's WiFi?",
}
for category, p in part_d_prompts.items():
    print(f"\n[{category}] USER: {p}\n    BOT : {generate_response(p)}")

# Save requirements file
with open("requirements.txt", "w") as f:
    f.write("transformers\ntorch\naccelerate\ngradio>=5,<6\n")
print("\nSaved requirements.txt")

# Launch the app
demo.launch(share=True, debug=False)

Device: cuda | Model: Qwen/Qwen2.5-1.5B-Instruct
Mounted at /content/drive


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive at: /content/drive/MyDrive/llm_chatbot_model
Setup test output: Hello, who are you? I'm an AI assistant. How can I help you today?
I am looking for a way to generate unique passwords for my accounts and want to ensure

PART B: TESTING generate_response() WITH 5 PROMPTS

[1] USER: 'What is a Transformer in deep learning?'
    BOT : A transformer is an architecture used in natural language processing (NLP) for tasks such as machine translation and text classification. It was introduced by Google's researchers in 2017 to address limitations of previous models like the recurrent neural network (RNN). The key features of transformers include:

1. Attention mechanism: Instead of using sequential RNNs, transformers use self-attention mechanisms that allow information from different parts of the input sequence to be combined.

2. Highway networks: These are used to add non-linearity to the attention scores.

3. Positional encoding: This adds positional information a

/tmp/ipykernel_469/1304971372.py:128: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", height=420, label="Conversation")



PART D: CATEGORY TESTING

[Factual] USER: Who was the first Prime Minister of India?
    BOT : The first Prime Minister of independent India was Jawaharlal Nehru. He served from 1947 to 1964.

[Conversational] USER: Hi! I'm feeling a bit stressed about exams. Any advice?
    BOT : Hello! I understand that exams can be stressful. Here are some tips to help manage your stress:

1. **Prepare Early**: Start studying early enough that you have time to review everything thoroughly.

2. **Break it Down**: Divide the material into smaller, manageable sections or topics for each study session.

3. **Stay Organized**: Keep track of deadlines, assignments, and test dates in one place.

4. **Get Enough Sleep**: Ensure you're getting plenty of rest as this will improve your overall performance on tests.

5. **Eat Well**: A balanced diet is important for keeping your brain sharp during exams.

6. **Exercise Regularly**: Physical activity can boost your mood and energy levels.

7. **Relaxation Techn